# Renewable-Powered Energy Management using Reinforcement Learning
### Team Daaku | CityLearn · SAC · PPO · Interactive Dashboard

This notebook is designed for **Google Colab**. It compares four controllers on a CityLearn building-energy simulation:

1. **No active battery control** (zero storage actions)
2. **Rule-based controller** (CityLearn BasicRBC)
3. **SAC** (primary RL agent)
4. **PPO** (comparison RL agent)

It records measured results from the environment; it does not pre-fill or fabricate performance numbers.

**Important scope note:** CityLearn's public challenge datasets may represent multiple buildings and may include assets beyond a single rooftop-solar building. This notebook inspects the selected dataset and reports metrics actually exposed by that dataset. If the project must strictly represent one building, configure a single-building schema after inspecting the available dataset schemas.

## 1. Install dependencies

Run this cell first. CityLearn's current documentation recommends its official package and documents Stable-Baselines3 wrappers. The installation may take a few minutes.

In [ ]:
%pip -q install "citylearn>=3.0.2,<3.1" "stable-baselines3>=2.3.2,<3" "gymnasium>=0.29,<1.2" "plotly>=5.18" "ipywidgets>=7.7" "tabulate>=0.9" pandas numpy

## 2. Imports and experiment settings

Start with a small training budget to verify the complete pipeline. Increase the training timesteps for final experiments and report the exact values used.

In [ ]:
import os, json, random, warnings, platform, sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

SEED = 42
# Quick smoke-test settings. Increase for final experiments.
TRAIN_TIMESTEPS = 20_000
EVAL_SEED = 2026
OUTPUT_DIR = Path("/content/renewable_energy_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

import citylearn
import stable_baselines3
from citylearn.citylearn import CityLearnEnv
from citylearn.agents.rbc import BasicRBC
from stable_baselines3 import SAC, PPO
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

print("Python:", sys.version.split()[0])
print("CityLearn:", getattr(citylearn, "__version__", "version attribute unavailable"))
print("Stable-Baselines3:", stable_baselines3.__version__)
print("Training timesteps per RL model:", TRAIN_TIMESTEPS)

## 3. Choose and inspect a public dataset

The default is an official CityLearn challenge dataset with building-energy time series. If the dataset name is unavailable in the installed release, the cell lists available datasets and selects a compatible challenge dataset when possible.

In [ ]:
from citylearn.data import DataSet

available_datasets = DataSet().get_dataset_names()
print(f"Available public datasets: {len(available_datasets)}")
display(pd.DataFrame({"dataset": available_datasets}).head(50))

PREFERRED_DATASET = "citylearn_challenge_2022_phase_all_plus_evs"
if PREFERRED_DATASET in available_datasets:
    DATASET_NAME = PREFERRED_DATASET
else:
    candidates = [x for x in available_datasets if "challenge" in x.lower() and "2022" in x.lower()]
    if not candidates:
        candidates = [x for x in available_datasets if "challenge" in x.lower()]
    if not candidates:
        raise RuntimeError("No compatible CityLearn challenge dataset found. Inspect available_datasets and choose a public dataset.")
    DATASET_NAME = candidates[0]

print("Selected dataset:", DATASET_NAME)

## 4. Environment helpers

Stable-Baselines3 requires a single-agent Gymnasium interface. CityLearn's official wrappers normalize observations and adapt the environment API. We create a **fresh environment for each controller** so one controller's run cannot contaminate another's results.

In [ ]:
def make_raw_env(seed=SEED, episode_time_steps=None):
    kwargs = {
        "schema": DATASET_NAME,
        "central_agent": True,
        "render_mode": "none",
        "random_seed": int(seed),
    }
    if episode_time_steps is not None:
        kwargs["episode_time_steps"] = int(episode_time_steps)
    return CityLearnEnv(**kwargs)

def make_sb3_env(seed=SEED):
    env = make_raw_env(seed=seed)
    env = NormalizedObservationWrapper(env)
    env = StableBaselines3Wrapper(env)
    return env

# Smoke test: one reset and one valid step.
_smoke_env = make_raw_env(seed=SEED)
_smoke_obs, _smoke_info = _smoke_env.reset(seed=SEED)
print("Observation type:", type(_smoke_obs))
print("Number of action spaces:", len(_smoke_env.action_space) if isinstance(_smoke_env.action_space, list) else "central action space")
print("Observation shape:", np.asarray(_smoke_obs, dtype=object).shape)
print("Building count:", len(getattr(_smoke_env, "buildings", [])))
print("Timesteps in environment:", getattr(_smoke_env, "time_steps", "not exposed"))
_smoke_env.close()

## 5. Controller 1 — no active battery control

This baseline submits zero-valued actions to the raw CityLearn action spaces. It is a no-active-control baseline, not necessarily a physically absent battery: the battery remains part of the simulation, but the controller does not request charge or discharge.

In [ ]:
def zero_action_for_space(space):
    return np.zeros(space.shape, dtype=np.float32)

def run_zero_control(seed=EVAL_SEED):
    env = make_raw_env(seed=seed)
    obs, info = env.reset(seed=seed)
    terminated = truncated = False
    steps = 0
    while not (terminated or truncated):
        if isinstance(env.action_space, list):
            actions = [zero_action_for_space(space) for space in env.action_space]
        else:
            actions = zero_action_for_space(env.action_space)
        obs, reward, terminated, truncated, info = env.step(actions)
        steps += 1
    kpis = env.evaluate_v2()
    env.close()
    return kpis, {"controller": "No active control", "steps": steps}

zero_kpis, zero_run_info = run_zero_control()
print("No-active-control run completed:", zero_run_info)
display(zero_kpis.head(12))

## 6. Controller 2 — rule-based baseline

CityLearn's BasicRBC is used as the documented rule-based controller. Its exact behaviour depends on the dataset's observations, actions and configured assets, so the report labels it as **CityLearn BasicRBC**, rather than claiming it is a custom tariff-optimized policy.

In [ ]:
def run_basic_rbc(seed=EVAL_SEED):
    env = make_raw_env(seed=seed)
    agent = BasicRBC(env)
    obs, info = env.reset(seed=seed)
    terminated = truncated = False
    steps = 0
    while not (terminated or truncated):
        actions = agent.predict(obs)
        obs, reward, terminated, truncated, info = env.step(actions)
        steps += 1
    kpis = env.evaluate_v2()
    env.close()
    return kpis, {"controller": "CityLearn BasicRBC", "steps": steps}

rbc_kpis, rbc_run_info = run_basic_rbc()
print("Rule-based run completed:", rbc_run_info)
display(rbc_kpis.head(12))

## 7. Train SAC and PPO

The initial training budget is intentionally configurable. A 20,000-timestep run is useful to test the pipeline, but it is **not evidence of convergence**. For final submission, compare multiple seeds and longer budgets if compute allows.

In [ ]:
def train_agent(algorithm, total_timesteps=TRAIN_TIMESTEPS, seed=SEED):
    env = make_sb3_env(seed=seed)
    if algorithm == "SAC":
        model = SAC(
            "MlpPolicy", env, seed=seed, verbose=1,
            learning_rate=3e-4, buffer_size=50_000,
            learning_starts=1_000, batch_size=256,
            train_freq=1, gradient_steps=1,
            policy_kwargs=dict(net_arch=[128, 128]),
        )
    elif algorithm == "PPO":
        model = PPO(
            "MlpPolicy", env, seed=seed, verbose=1,
            learning_rate=3e-4, n_steps=1024, batch_size=64,
            n_epochs=10, gamma=0.99,
            policy_kwargs=dict(net_arch=dict(pi=[128, 128], vf=[128, 128])),
        )
    else:
        raise ValueError("algorithm must be SAC or PPO")
    model.learn(total_timesteps=int(total_timesteps), progress_bar=True)
    model_path = OUTPUT_DIR / f"{algorithm.lower()}_citylearn"
    model.save(str(model_path))
    env.close()
    print(f"Saved {algorithm} model to {model_path}")
    return model

sac_model = train_agent("SAC")
ppo_model = train_agent("PPO")

## 8. Evaluate learned policies on a fresh environment

Each agent is evaluated deterministically on a fresh environment using the same evaluation seed. CityLearn's KPI table is retained in full so the selected dataset's actual metric names remain visible.

In [ ]:
def evaluate_sb3_model(model, label, seed=EVAL_SEED):
    env = make_sb3_env(seed=seed)
    obs, info = env.reset(seed=seed)
    terminated = truncated = False
    steps = 0
    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        steps += 1
    raw_env = env.unwrapped
    kpis = raw_env.evaluate_v2()
    env.close()
    return kpis, {"controller": label, "steps": steps}

sac_kpis, sac_run_info = evaluate_sb3_model(sac_model, "SAC")
ppo_kpis, ppo_run_info = evaluate_sb3_model(ppo_model, "PPO")

print(sac_run_info)
print(ppo_run_info)

## 9. Normalize CityLearn KPI outputs for comparison

CityLearn KPI names vary by dataset/version. This cell preserves all original KPI rows and builds a compact table from recognized cost, grid-import, peak, renewable, and carbon metric names. Missing metrics stay missing; they are not replaced with zero.

In [ ]:
def add_controller_name(df, name):
    out = df.copy()
    out["controller"] = name
    return out

all_kpis = pd.concat([
    add_controller_name(zero_kpis, "No active control"),
    add_controller_name(rbc_kpis, "CityLearn BasicRBC"),
    add_controller_name(sac_kpis, "SAC"),
    add_controller_name(ppo_kpis, "PPO"),
], ignore_index=True)

display(all_kpis.head(30))
all_kpis.to_csv(OUTPUT_DIR / "citylearn_all_kpis.csv", index=False)

def choose_metric_rows(df):
    # Select useful KPI families without assuming every dataset has all metrics.
    patterns = {
        "Cost": ["cost_total", "electricity_cost", "cost"],
        "Grid import": ["grid_total_import", "grid_import", "electricity_consumption"],
        "Peak demand": ["peak", "ramping"],
        "Renewable / self-consumption": ["self_consumption", "solar", "renewable"],
        "Carbon emissions": ["carbon", "emission"],
        "Service / unmet demand": ["unserved", "unmet", "service_violations", "outage"],
    }
    rows = []
    for family, tokens in patterns.items():
        mask = all_kpis["cost_function"].astype(str).str.lower().apply(
            lambda s: any(token in s for token in tokens)
        ) if "cost_function" in all_kpis.columns else pd.Series(False, index=all_kpis.index)
        subset = all_kpis[mask].copy()
        if not subset.empty and "value" in subset.columns:
            subset["metric_family"] = family
            rows.append(subset)
    if rows:
        return pd.concat(rows, ignore_index=True)
    return pd.DataFrame()

selected_kpis = choose_metric_rows(all_kpis)
if selected_kpis.empty:
    print("No KPI names matched the automatic metric-family selection. Inspect all_kpis and select appropriate rows for this dataset.")
else:
    display(selected_kpis[["controller", "cost_function", "value"]].drop_duplicates().sort_values(["cost_function", "controller"]))
    selected_kpis.to_csv(OUTPUT_DIR / "selected_kpi_comparison.csv", index=False)

## 10. Compute savings only from observed cost metrics

This helper finds cost-related KPI rows and computes relative change against the no-active-control baseline only when matching KPI names exist. The sign convention is explicit: positive savings means the controller's measured cost is lower than the baseline.

In [ ]:
def compute_cost_savings(kpis_long):
    if kpis_long.empty or not {"controller", "cost_function", "value"}.issubset(kpis_long.columns):
        return pd.DataFrame()
    data = kpis_long.copy()
    data["cost_function"] = data["cost_function"].astype(str)
    cost_rows = data[data["cost_function"].str.lower().str.contains("cost_total|electricity_cost", regex=True)].copy()
    if cost_rows.empty:
        return pd.DataFrame()
    # If there are multiple cost KPI rows, retain each explicitly rather than silently summing them.
    wide = cost_rows.pivot_table(index="cost_function", columns="controller", values="value", aggfunc="first")
    if "No active control" not in wide.columns:
        return wide.reset_index()
    for controller in ["CityLearn BasicRBC", "SAC", "PPO"]:
        if controller in wide.columns:
            wide[f"{controller} savings (%)"] = (
                (wide["No active control"] - wide[controller]) / wide["No active control"].replace(0, np.nan) * 100
            )
    return wide.reset_index()

cost_comparison = compute_cost_savings(all_kpis)
display(cost_comparison if not cost_comparison.empty else pd.DataFrame({"note": ["No directly comparable total-cost KPI was detected. Inspect all_kpis before claiming savings."]}))
cost_comparison.to_csv(OUTPUT_DIR / "cost_comparison.csv", index=False)

## 11. Interactive dashboard

Use the dropdowns to inspect controller KPIs. The dashboard visualizes only recorded CityLearn values. It also shows the run configuration so results can be interpreted in context.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

controller_dropdown = widgets.Dropdown(
    options=["No active control", "CityLearn BasicRBC", "SAC", "PPO"],
    value="SAC",
    description="Controller:",
    layout=widgets.Layout(width="360px"),
)
metric_search = widgets.Text(
    value="",
    placeholder="Filter KPI names (e.g. cost, grid, carbon)",
    description="Filter:",
    layout=widgets.Layout(width="520px"),
)
dashboard_out = widgets.Output()

def render_dashboard(*args):
    with dashboard_out:
        clear_output(wait=True)
        name = controller_dropdown.value
        data = all_kpis[all_kpis["controller"] == name].copy()
        if "cost_function" in data.columns:
            data["cost_function"] = data["cost_function"].astype(str)
            query = metric_search.value.strip().lower()
            if query:
                data = data[data["cost_function"].str.lower().str.contains(query, regex=False)]
            if "value" in data.columns and not data.empty:
                data["value"] = pd.to_numeric(data["value"], errors="coerce")
                data = data.dropna(subset=["value"])
                if len(data):
                    fig = px.bar(
                        data.sort_values("value", ascending=False).head(25),
                        x="cost_function", y="value",
                        title=f"Measured CityLearn KPIs — {name}",
                        hover_data=[c for c in ["level", "name", "unit"] if c in data.columns],
                    )
                    fig.update_layout(xaxis_tickangle=-45, height=520, margin=dict(b=160))
                    fig.show()
                else:
                    print("No numeric KPI rows match this filter.")
            else:
                print("No KPI data available for this controller.")
        else:
            print("The KPI table has no cost_function field; inspect all_kpis.")
        display(data)

controller_dropdown.observe(render_dashboard, names="value")
metric_search.observe(render_dashboard, names="value")
display(widgets.VBox([controller_dropdown, metric_search, dashboard_out]))
render_dashboard()

## 12. Optional: compare selected metrics across controllers

This chart uses one row per controller and metric. Because CityLearn may expose several cost functions, choose the exact KPI to compare instead of aggregating unrelated metrics.

In [ ]:
if not all_kpis.empty and {"controller", "cost_function", "value"}.issubset(all_kpis.columns):
    metric_options = sorted(all_kpis["cost_function"].dropna().astype(str).unique().tolist())
    metric_picker = widgets.Dropdown(
        options=metric_options,
        description="Exact KPI:",
        layout=widgets.Layout(width="700px"),
    )
    metric_plot_out = widgets.Output()

    def plot_exact_metric(change=None):
        with metric_plot_out:
            clear_output(wait=True)
            metric = metric_picker.value
            data = all_kpis[all_kpis["cost_function"].astype(str) == metric].copy()
            data["value"] = pd.to_numeric(data["value"], errors="coerce")
            data = data.dropna(subset=["value"])
            if not data.empty:
                fig = px.bar(data, x="controller", y="value", title=f"Controller comparison: {metric}", text_auto=".4g")
                fig.update_layout(height=420)
                fig.show()
                display(data)
            else:
                print("No numeric values for this KPI.")
    metric_picker.observe(plot_exact_metric, names="value")
    display(widgets.VBox([metric_picker, metric_plot_out]))
    plot_exact_metric()
else:
    print("KPI schema not recognized; inspect all_kpis.")

## 13. Export models, metrics, and a machine-generated report

The report states which runs were executed and includes measured KPI tables. It deliberately avoids a claim that RL outperforms the baselines unless the recorded values support it.

In [ ]:
run_summary = pd.DataFrame([
    {"controller": "No active control", **zero_run_info},
    {"controller": "CityLearn BasicRBC", **rbc_run_info},
    {"controller": "SAC", **sac_run_info},
    {"controller": "PPO", **ppo_run_info},
])
run_summary = run_summary.loc[:, ~run_summary.columns.duplicated()]
run_summary.to_csv(OUTPUT_DIR / "run_summary.csv", index=False)

report_path = OUTPUT_DIR / "experiment_report.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("# Renewable-Powered Energy Management — Experiment Report\n\n")
    f.write("## Configuration\n")
    f.write(f"- Dataset: `{DATASET_NAME}`\n")
    f.write(f"- CityLearn version: `{getattr(citylearn, '__version__', 'unknown')}`\n")
    f.write(f"- Stable-Baselines3 version: `{stable_baselines3.__version__}`\n")
    f.write(f"- Training timesteps per RL agent: `{TRAIN_TIMESTEPS}`\n")
    f.write(f"- Training seed: `{SEED}`; evaluation seed: `{EVAL_SEED}`\n\n")
    f.write("## Runs\n\n")
    f.write(run_summary.to_markdown(index=False))
    f.write("\n\n## Measured CityLearn KPIs\n\n")
    if not selected_kpis.empty:
        f.write(selected_kpis[["controller", "cost_function", "value"]].to_markdown(index=False))
    else:
        f.write("No KPI rows were automatically selected. See `citylearn_all_kpis.csv` for the complete measured KPI output.\n")
    f.write("\n\n## Interpretation and limitations\n")
    f.write("- Values are generated by this notebook's actual simulation runs; they are not pre-filled benchmark claims.\n")
    f.write("- A single seed and short training budget are not sufficient to establish generalization or statistical significance.\n")
    f.write("- Verify that the selected dataset and asset configuration match the project's intended single-building solar-plus-battery scenario.\n")
    f.write("- Only metrics provided by the selected CityLearn dataset/schema should be reported as measured.\n")

print("Saved outputs:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)
print("\nReport preview:\n")
print(report_path.read_text(encoding="utf-8")[:5000])

## 14. Download the experiment outputs

This creates a ZIP containing the CSV results, Markdown report, and saved SAC/PPO models. In Colab, run the cell to download it.

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/renewable_energy_project_outputs", "zip", OUTPUT_DIR)
print("ZIP created:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not running in Google Colab. ZIP remains at:", zip_path)

## 15. Recommended final-experiment checklist

- [ ] Increase training timesteps and document compute/time constraints.
- [ ] Run at least 3 random seeds for SAC and PPO where feasible.
- [ ] Keep training and evaluation periods separate; do not tune on the held-out test period.
- [ ] Verify the selected schema has rooftop PV and battery storage, and document the building count.
- [ ] Confirm how each metric is defined in the CityLearn KPI reference before interpreting it.
- [ ] Report raw measured values as well as percentage changes, with metric units.
- [ ] Do not claim unmet demand or shortage hours were measured unless the selected schema exposes a valid service/shortage metric.
- [ ] Include training configuration, package versions, seeds, limitations, and the exact dataset name in the submission.